# Assignment 2 — Diversification, and Your First Sort


Graded on completion. Work in your group of three; each of you submits your own copy.

---

**Group members**

- Name / NYU email:
- Name / NYU email:
- Name / NYU email:

---

Three parts. **Part 1** is the same for everybody and answers a question Lecture 2
deliberately left open. **Part 2**, also the same for everybody, walks you through
NYSE breakpoints — the convention research papers use to build sorted
portfolios — and what that one choice does to two famous results. **Part 3** is
where your group's own strategy starts — everything you do for the rest of the
term builds on it.

Use AI freely. You will be asked to explain what you did.

## 🛠️ Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = [10, 4]
import warnings; warnings.filterwarnings('ignore')

BASE  = "https://raw.githubusercontent.com/amoreira2/UG54/refs/heads/main/assets/data"
panel = pd.read_parquet(f"{BASE}/panel_backbone_1980_2000.parquet")
menu  = pd.read_csv(f"{BASE}/signal_menu.csv")

print(f"{len(panel):,} stock-months, {panel.permno.nunique():,} stocks, "
      f"{panel.date.nunique()} months")
print(f"{len(menu)} signals on the menu")

---

# Part 1 — How much does diversification actually buy you?

In Lecture 2 we found that a typical stock runs about **52% volatility a year**
while the whole market runs **15.5%**, and left two questions hanging. This is
the first one:

> There are roughly 6,000 stocks. If they moved independently, averaging them
> would drive volatility to almost nothing. It stops at 15.5%. **Why?**

You are going to answer it by measuring, not by being told.

### Q1 — A clean set of stocks to work with

Comparing portfolios of different sizes only makes sense if every portfolio
covers the same months. Build a **wide** table — dates down the rows, one column
per stock, returns in the cells — keeping only stocks that are present for the
**entire** sample.

> **📝 Spec.** From `panel`, produce a DataFrame `W` indexed by `date` with one
> column per `permno`, containing `ret`. Drop any stock that has a missing month.
> Report how many stocks survive.

> **🤖 A prompt that gets you close:** *"Pivot a long DataFrame with columns
> permno, date, ret into a wide DataFrame indexed by date with one column per
> permno, then drop every column that contains any NaN."*

In [ ]:
# Your work here


W = ____          # wide table: dates x stocks
print(f"{W.shape[1]} stocks with a complete {W.shape[0]}-month history")

### Q2 — The volatility of an equal-weighted portfolio of N stocks

Write a function that takes a number of stocks `N`, picks `N` of them **at
random** from `W`, forms the equal-weighted portfolio, and returns its
**annualized** volatility.

Two things to be careful about. An equal-weighted portfolio of a set of columns
is just their average, row by row. And annualizing a monthly volatility means
multiplying by √12, not by 12.

In [ ]:
rng = np.random.default_rng(0)      # so your answers are reproducible

def port_vol(N):
    """Annualized volatility of an equal-weighted portfolio of N random stocks."""
    ____

print(f"one draw of 10 stocks: {port_vol(10):.1%}")

### Q3 — Average over many draws

One draw is noise — you might happen to pick ten utilities. Write
`avg_port_vol(N, n_draws=100)` that repeats Q2 `n_draws` times and returns the
average.

In [ ]:
def avg_port_vol(N, n_draws=100):
    ____

print(f"N=10, averaged over 100 draws: {avg_port_vol(10):.1%}")

### Q4 — The curve

Compute `avg_port_vol` for **N = 1, 2, 5, 10, 20, 50, 100, 200** and plot it
against N. Put N on a log scale — the interesting action is at small N.

In [ ]:
NS = [1, 2, 5, 10, 20, 50, 100, 200]

# Your work here


### Q5 — Two reference lines

The plot on its own does not tell you what it *should* have looked like. Add
two lines.

**Line A — what you would get if stocks were independent.** If N stocks each had
volatility σ and were uncorrelated, the equal-weighted portfolio's volatility
would be σ/√N. Use the average single-stock volatility in `W` for σ, and draw
that curve across the same range of N.

**Line B — the market.** Compute the value-weighted market return from `panel`
(you built this in Lecture 2: weight by `me` lagged one month, earn this
month's `ret`) and draw its annualized volatility as a horizontal line.

> **⚠️ Careful.** Line A is *not* a fitted line or a prediction. It is what the
> arithmetic gives you under an assumption you have every reason to doubt. The
> point of drawing it is to see how badly the assumption fails.

In [ ]:
# Your work here


### Q6 — Read your own plot

> **📝 Answer in the cell below — a short paragraph each, no code.**
>
> **(a)** Roughly where does your curve stop falling? Give an N.
>
> **(b)** The independent line keeps going down and your curve does not. Describe
> the gap between them at N = 200 and say, in your own words, what is in that
> gap. What is it about real stocks that the independent calculation ignores?
>
> **(c)** Look at where your curve levels off relative to the market line. Is
> that a coincidence? What would have to be true for a large equal-weighted
> portfolio to end up at the market's volatility?
>
> **(d)** You are about to build a long-short portfolio out of roughly 600
> stocks per leg. Given this plot, what does that buy you — and what does it
> *not* protect you from?

**(a)**

**(b)**

**(c)**

**(d)**

---

# Part 2 — NYSE breakpoints, and why the details decide the answer

In the sorts lecture you cut stocks into ten equal-count buckets with `pd.qcut`,
over every stock in the market. Research papers almost never do that. They use
**NYSE breakpoints**. This part builds them one step at a time, then uses them on
one of the oldest results in finance — that small stocks outperform (Banz, 1981)
— and on value.

Most of the code is written for you. Run each cell, read what it prints, and make
sure you can say what every line does. The questions are where you do the work.

**Why equal-count buckets are a problem.** There are about 6,000 stocks, and the
ten largest are a fifth of the market's value. Size is so skewed that an
equal-count bottom decile is roughly 600 of the smallest companies in the
country: firms worth a few million dollars, many of which barely trade.

**The fix.** Compute the cutoffs from **NYSE-listed stocks only**, `exchcd == 1`,
then apply them to **every** stock. NYSE firms are larger and more established,
so "small" is defined relative to them rather than relative to the microcap
tail. Fama and French do this, and so does almost every published paper.

The timing is the lecture's: everything you condition on is last month's value,
and the return you earn is this month's `ret`.

### Step 1 — Last month's size

In [ ]:
panel['me_l1'] = panel.groupby('permno')['me'].shift(1)
d0 = panel.dropna(subset=['ret', 'me_l1'])

print(f"{len(d0):,} stock-months with a return and last month's size")
print(f"stocks per month: {d0.groupby('date').size().mean():,.0f} in all, "
      f"{d0[d0.exchcd == 1].groupby('date').size().mean():,.0f} of them on the NYSE")

### Step 2 — The cutoffs, from NYSE stocks only

For each month, the 10th and 90th percentile of `me_l1` among NYSE stocks. Read
the chain one piece at a time:

| Piece | What it does |
|---|---|
| `d0[d0.exchcd == 1]` | keeps NYSE stocks only |
| `.groupby('date')['me_l1']` | one pile of market caps per month |
| `.quantile([.1, .9])` | the 10th and 90th percentile of each pile: two numbers per month |
| `.unstack()` | turns those two numbers into two columns |
| `.rename(columns={0.1: 'lo', 0.9: 'hi'})` | gives the columns names |

In [ ]:
q = (d0[d0.exchcd == 1].groupby('date')['me_l1']
       .quantile([.1, .9]).unstack().rename(columns={0.1: 'lo', 0.9: 'hi'}))

print(f"{len(q)} months of cutoffs, in $ millions:\n")
print((q / 1000).round(1).head(3).to_string())      # me is in $ thousands

In February 1980 a stock counts as small below about \$33 million and as big
above about \$1.3 billion. Those cutoffs come from NYSE stocks alone. Nothing has
been applied to anybody yet.

### Step 3 — Apply the cutoffs to every stock

`merge` on `date` copies each month's `lo` and `hi` onto every stock in that
month — NYSE, AMEX and NASDAQ alike. Then `np.where` labels each stock-month.

`np.where(condition, a, b)` goes row by row: where the condition is true it
takes `a`, otherwise `b`. A second `np.where` in the `b` slot gives three
outcomes: `'S'` at or below `lo`, `'B'` at or above `hi`, and `None` — Python's
"nothing" — for everything in between.

In [ ]:
a = d0.merge(q, on='date')
a['g'] = np.where(a.me_l1 <= a.lo, 'S', np.where(a.me_l1 >= a.hi, 'B', None))

print(a['g'].value_counts(dropna=False).to_string(), "\n")    # stock-months with each label
a[['permno', 'date', 'exchcd', 'me_l1', 'lo', 'hi', 'g']].head()

### Q7 — How many stocks are "small"?

"Compute from NYSE, apply to every stock" is two clauses, and it is easy to merge
them into one. Count the stocks in the small bucket each month, averaged over
the months, under three readings:

1. **Cutoffs from NYSE, applied to every stock** — what Step 3 built.
2. **Cutoffs from NYSE, applied to NYSE stocks only** — the same labels, keeping
   only `exchcd == 1`.
3. **No breakpoints** — `pd.qcut` into ten equal-count buckets over every stock
   each month, as in the lecture, counting the bottom bucket.

> **🤖 A prompt that gets you close for reading 1:** *"From DataFrame `a`, keep
> the rows where `g == 'S'`, count the rows for each `date`, and take the mean of
> those counts."*

> **📌 Name your answers:**
> ```python
> n_small_nyse  = ____   # reading 1
> n_small_wrong = ____   # reading 2
> n_small_all   = ____   # reading 3
> ```

In [ ]:
# Your work here


n_small_nyse  = ____
n_small_wrong = ____
n_small_all   = ____

print(f"{'cutoffs from NYSE, applied to every stock':44s}{n_small_nyse:>7,.0f}")
print(f"{'cutoffs from NYSE, applied to NYSE only':44s}{n_small_wrong:>7,.0f}")
print(f"{'no breakpoints: qcut over every stock':44s}{n_small_all:>7,.0f}")

> **✅ Check.** You should get about **3,100**, **150** and **600**. If you
> don't, go back to the three readings before moving on — everything after this
> depends on it.

**Q7, in words.** Two or three sentences. Why does "small relative to NYSE" take
in half the market, when "the smallest tenth of everything" is 600 stocks? And
what are the 150 stocks of reading 2 — what kind of company ends up in a "small"
portfolio built that way?

**Q7:**

### Step 4 — The size effect, one way

Now earn returns. Keep the two legs, value-weight each one by `me_l1` within
each month, and subtract: small minus big. The `groupby` is on two columns,
month and leg, and `.unstack()` puts the two legs side by side — the same moves
as the lecture's decile table.

In [ ]:
legs = a[a.g.notna()]                                   # small and big only
vw = (legs.groupby(['date', 'g'])
          .apply(lambda x: np.average(x['ret'], weights=x['me_l1'])).unstack())
smb = (vw['S'] - vw['B']).dropna()                       # small minus big, each month

print(f"small minus big, NYSE breakpoints, value-weighted: "
      f"{smb.mean()*12:+.1%}/yr   t = {smb.mean()/smb.std()*np.sqrt(len(smb)):.2f}")

### Step 5 — Four defensible versions

The function below is Steps 2 to 4 with two switches: `breakpoints='nyse'` or
`'all'` (equal-count `qcut` over every stock, as in the lecture), and
`weighting='vw'` or `'ew'`. Every line is a step you have already run. It hands
back four numbers: the annualized mean, the annualized volatility, the
t-statistic, and the average number of stocks in the small leg.

Two pieces of Python you haven't seen: `if` / `else` runs one indented block or
the other, depending on whether the condition is true, and `a if condition else b`
is the same choice written in one line.

In [ ]:
def size_spread(breakpoints, weighting):
    """Small minus big, 1980-2000. Returns (annualized mean, vol, t, n_small)."""
    d = d0.copy()
    if breakpoints == 'nyse':
        q = (d[d.exchcd == 1].groupby('date')['me_l1']
               .quantile([.1, .9]).unstack().rename(columns={0.1: 'lo', 0.9: 'hi'}))
        d = d.merge(q, on='date')
        d['g'] = np.where(d.me_l1 <= d.lo, 'S', np.where(d.me_l1 >= d.hi, 'B', None))
    else:
        dec = d.groupby('date')['me_l1'].transform(
            lambda x: pd.qcut(x, 10, labels=False, duplicates='drop'))
        d['g'] = np.where(dec == 0, 'S', np.where(dec == 9, 'B', None))
    d = d[d.g.notna()]
    agg = ((lambda g: np.average(g['ret'], weights=g['me_l1'])) if weighting == 'vw'
           else (lambda g: g['ret'].mean()))
    p = d.groupby(['date', 'g']).apply(agg).unstack()
    r = (p['S'] - p['B']).dropna()
    n = d[d.g == 'S'].groupby('date').size().mean()
    return r.mean()*12, r.std()*np.sqrt(12), r.mean()/r.std()*np.sqrt(len(r)), n

m, v, t, n = size_spread('nyse', 'vw')
print(f"NYSE, value-weighted: {m:+.1%}/yr, vol {v:.1%}, t = {t:.2f}, {n:,.0f} stocks small  <- Step 4 again")

### Q8 — The four versions

Call `size_spread` four times — all-stock deciles and NYSE breakpoints, each
equal- and value-weighted — and keep the four t-statistics.

> **🤖 Hint:** `m, v, t, n = size_spread('all', 'ew')` unpacks the four numbers
> the function returns.

> **📌 Name your answers:**
> ```python
> t_ew_all  = ____   # all-stock deciles, equal-weighted
> t_vw_all  = ____   # all-stock deciles, value-weighted
> t_ew_nyse = ____   # NYSE breakpoints, equal-weighted
> t_vw_nyse = ____   # NYSE breakpoints, value-weighted
> ```

In [ ]:
# Your work here


t_ew_all  = ____
t_vw_all  = ____
t_ew_nyse = ____
t_vw_nyse = ____

print("SMALL MINUS BIG, 1980-2000 — t-statistics")
print(f"  all-stock deciles, equal-weighted : {t_ew_all:+.2f}")
print(f"  all-stock deciles, value-weighted : {t_vw_all:+.2f}")
print(f"  NYSE breakpoints,  equal-weighted : {t_ew_nyse:+.2f}")
print(f"  NYSE breakpoints,  value-weighted : {t_vw_nyse:+.2f}")

> **✅ Check.** From about **+3.9** down to about **−1.9**. The same stocks, the
> same months, the same idea of "small" — and the answer runs from one of the
> great anomalies in finance to the opposite.

### Step 6 — Value, under the standard convention

The lecture's value demo sorted on `BM` with equal-count deciles and equal
weights. Here is that version next to NYSE breakpoints and value weights, the
convention the rest of the course uses.

In [ ]:
bm = pd.read_parquet(f"{BASE}/signals/BM.parquet")
val = panel.merge(bm, on=['permno', 'date'], how='left').sort_values(['permno', 'date'])
val['BM_l1'] = val.groupby('permno')['BM'].shift(1)
val = val.dropna(subset=['BM_l1', 'ret', 'me_l1'])

# the lecture's version: equal-count deciles, equal-weighted
val['dec'] = val.groupby('date')['BM_l1'].transform(
    lambda x: pd.qcut(x, 10, labels=False, duplicates='drop'))
pe = val.groupby(['date', 'dec'])['ret'].mean().unstack()
hml_lecture = (pe[9] - pe[0]).dropna()

# the standard convention: NYSE breakpoints, value-weighted
qv = (val[val.exchcd == 1].groupby('date')['BM_l1']
        .quantile([.1, .9]).unstack().rename(columns={0.1: 'lo', 0.9: 'hi'}))
val = val.merge(qv, on='date')
val['g'] = np.where(val.BM_l1 <= val.lo, 'L', np.where(val.BM_l1 >= val.hi, 'H', None))
pv = (val[val.g.notna()].groupby(['date', 'g'])
          .apply(lambda x: np.average(x['ret'], weights=x['me_l1'])).unstack())
hml_nyse = (pv['H'] - pv['L']).dropna()

for label, r in [('equal-count deciles, equal-weighted', hml_lecture),
                 ('NYSE breakpoints,   value-weighted', hml_nyse)]:
    print(f"value, {label} : {r.mean()*12:+.1%}/yr   t = {r.mean()/r.std()*np.sqrt(len(r)):.2f}")

### Q9 — What the details did

> **📝 Answer in the cell below — a short paragraph each, no code.**
>
> **(a)** Size: how far apart are your four answers, and does the sign change?
> Nothing about the data changed between them. What did?
>
> **(b)** If your PM asked "do small stocks outperform?", which of the four would
> you report, and why?
>
> **(c)** What drives the gap between the two ends of the table? Use your answer
> to Q7.
>
> **(d)** In what sense is the equal-weighted, all-stock number real, and in what
> sense is it not?
>
> **(e)** Value shrinks by a factor of four between the two lines of Step 6.
> Which two choices account for that?

**(a)**

**(b)**

**(c)**

**(d)**

**(e)**

---

# Part 3 — Your strategy

From here on this is your group's project. Whatever you pick now, you will carry
through the term — you may change it later, but pick something you find
interesting enough to argue about.

### Q10 — Pick a signal, and commit to a story first

Look at the menu. Pick **one** signal.

Then, **before you compute anything**, write two or three sentences on why it
might predict returns. Is it compensation for a risk somebody is unwilling to
bear, or is it a mistake other investors are making? You are not being graded on
being right. You are being graded on having said something falsifiable before
you saw the answer.

In [ ]:
pd.set_option('display.max_rows', 40, 'display.width', 200)
menu[['Acronym','Authors','Year','Cat.Economic','T-Stat']].sort_values('Cat.Economic')

In [ ]:
MY_SIGNAL = "____"        # ← your pick

sig = pd.read_parquet(f"{BASE}/signals/{MY_SIGNAL}.parquet")
row = menu.loc[menu.Acronym == MY_SIGNAL].iloc[0]
print(f"{MY_SIGNAL} — {row.Authors} ({row.Year}), published t = {row['T-Stat']:+.2f}")
print(f"{len(sig):,} stock-months\n")
print(row.LongDescription[:400])

**Why might this signal predict returns?** *(write before you run anything)*



### Q11 — Run the standard sort

Everyone in the class uses the same convention so results are comparable:
**NYSE breakpoints, value-weighted, top decile minus bottom decile.** As in
Lecture 2, everything you condition on is lagged one month and the return you
earn is this month's `ret`.

> **📝 Spec.** Merge your signal onto `panel`, keeping every panel month
> (`how='left'`), then build two lagged columns:
>
> ```python
> d['me_l1']  = d.groupby('permno')['me'].shift(1)
> d['sig_l1'] = d.groupby('permno')[MY_SIGNAL].shift(1)
> ```
>
> Drop rows missing `ret`, `me_l1` or `sig_l1`, **before** ranking. Each month,
> compute the 10th and 90th percentile of `sig_l1` **using NYSE stocks only**
> (`exchcd == 1`), then apply those cutoffs to **every** stock. Value-weight
> within each leg using `me_l1`. Return the monthly series of top-minus-bottom.
>
> **Lag the signal, not just the weights.** Sorting on `sig` at *t* while
> earning `ret` at *t* is look-ahead — you would be picking stocks using a
> number you could not have known.

Part 2 built exactly this for size. Reuse it with your signal as the sorting
variable and `me_l1` as the weights — and make sure you can say what each line
does.

In [ ]:
# Your work here


ls = ____        # monthly long-short return series
print(f"{len(ls)} months")

### Q12 — Report it honestly

Report, for your long-short: annualized mean return, annualized volatility,
Sharpe ratio, and the t-statistic. Then report **each leg separately** — a
spread built entirely from a collapsing short leg is a different claim from one
where both sides contribute. Finally, put your t-statistic next to the one the
original authors published.

In [ ]:
# Your work here


### Q13 — Look at the shape, not just the spread

Plot the annualized mean return of all ten deciles as a bar chart.

A **monotone** staircase — each decile beating the one below — is much stronger
evidence than a large gap between two extremes with a flat middle. The second
pattern usually means two small groups of unusual firms are doing all the work.

In [ ]:
# Your work here


### Q14 — The memo

> **📝 Maximum eight sentences. This is the part I will actually read.**
>
> Your PM asks whether this signal is worth pursuing. Answer them.
>
> Cover: what you found; whether it replicated the published result and what you
> make of it if it didn't; whether the sort is monotone; which leg is doing the
> work; and one specific thing you would want to check before putting money on
> it.
>
> If your signal did not work, say so plainly. **Most signals do not work, and a
> project that establishes that honestly is a good project.** You will not be
> penalised for a negative result — only for pretending you didn't get one.

**Memo:**

